# Chuẩn bị video mới trên server CPU

**Flow:** `video/` → keyframes + mapping → SigLIP features → object detection → BLIP caption/VQA → caption mapping + FAISS index + manifest → validation.

Chọn kernel `.venv/bin/python` của **server Remote-SSH**, rồi chạy từ trên xuống. Các bước inference đều ép `device='cpu'`. Không cần Gemini API key hay GPU cho notebook này.

Trước lần chạy đầu, trong terminal remote: `source .venv/bin/activate` và `python -m pip install -r requirements-video-cpu.txt` (hoặc `uv pip install --python .venv/bin/python -r requirements-video-cpu.txt`). Video phải có tên `Lxx_Vxxx.mp4` để tương thích bộ retriever hiện tại.

## 1. Cấu hình và xác nhận kernel

Lần đầu để `SMOKE_MODE=True`: xử lý 12 giây vào thư mục riêng `cpu_smoke/random_80_160_seed_42/`. Sau khi kiểm tra OK, đổi sang `False` và chạy lại để build toàn bộ video vào các thư mục chính.

`VIDEO_IDS=None` xử lý tất cả video trong `video/`, kể cả các folder con. Batch nhỏ và giới hạn số threads giúp giảm RAM/tải trên server dùng chung.

In [ ]:
from pathlib import Path
import os
import socket
import sys

PROJECT_DIR = Path.cwd().resolve()  # Server trường: /home/team09/KISBranche
EXPECTED_HOSTNAME = 'ubuntu'       # Sửa theo hostname server của bạn; None để bỏ kiểm tra tên
VIDEO_DIR = PROJECT_DIR / 'video'
VIDEO_IDS = ['L20_V001']           # None để xử lý tất cả video
CPU_THREADS = min(8, os.cpu_count() or 1)
BATCH_SIZE = 2
CAPTION_BATCH_SIZE = 1
KEYFRAME_MIN_GAP = 80             # Số frame tối thiểu giữa hai ảnh
KEYFRAME_MAX_GAP = 160            # Số frame tối đa giữa hai ảnh (inclusive)
RANDOM_SEED = 42                  # Cùng seed/video cho kết quả giống nhau khi chạy lại
SMOKE_MODE = True
SMOKE_SECONDS = 12.0
OUTPUT_ROOT = (PROJECT_DIR / 'cpu_smoke' / f'random_{KEYFRAME_MIN_GAP}_{KEYFRAME_MAX_GAP}_seed_{RANDOM_SEED}') if SMOKE_MODE else PROJECT_DIR

if sys.platform != 'linux':
    raise RuntimeError('Hãy chọn kernel Linux trên server trường trước khi chạy notebook.')
if EXPECTED_HOSTNAME and socket.gethostname() != EXPECTED_HOSTNAME:
    raise RuntimeError(f'Kernel ở {socket.gethostname()}, khác server {EXPECTED_HOSTNAME}.')
if not (PROJECT_DIR / 'prepare_video_data.py').is_file():
    raise RuntimeError('Đặt PROJECT_DIR thành thư mục repository trên server.')
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))
from prepare_video_data import configure_cpu, discover_videos, KeyframeConfig, VideoDatasetBuilder
print(configure_cpu(CPU_THREADS))
print('Output:', OUTPUT_ROOT)
print('Smoke mode:', SMOKE_MODE)

In [ ]:
videos = discover_videos(VIDEO_DIR, VIDEO_IDS)
builder = VideoDatasetBuilder(OUTPUT_ROOT)
keyframe_config = KeyframeConfig(
    frame_gap_min=KEYFRAME_MIN_GAP,
    frame_gap_max=KEYFRAME_MAX_GAP,
    random_seed=RANDOM_SEED,
    max_duration_seconds=SMOKE_SECONDS if SMOKE_MODE else None,
)
for video_id, path in videos.items():
    print(video_id, '->', path, f'({path.stat().st_size / 1024**2:.1f} MiB)')

## 2. Tách keyframes và mapping

Giữ frame 0; sau mỗi ảnh chọn ngẫu nhiên khoảng cách 80–160 frame để lấy ảnh tiếp theo. Seed kết hợp video_id giúp chạy lại ra cùng kết quả. Chế độ này lấy mẫu theo số frame, không thêm ảnh tại chuyển cảnh. `frame_idx` là vị trí frame decode thật (0-based); `pts_time` dùng presentation timestamp khi video có PTS, kể cả VFR.

Ảnh: `data/<group>/<video_id>/001.jpg`; CSV: `mapping/<video_id>.csv` với `n,pts_time,fps,frame_idx`. Checkpoint/checksum nằm trong `preparation/`. Chạy lại sẽ resume cùng source/config; nếu video_id đã có artifact do provider tạo thì bước này từ chối ghi đè.

In [ ]:
for video_id, video_path in videos.items():
    rows = builder.extract_keyframes(video_path, keyframe_config)
    frame_ids = [row['frame_idx'] for row in rows]
    gaps = [current - previous for previous, current in zip(frame_ids, frame_ids[1:])]
    assert all(KEYFRAME_MIN_GAP <= gap <= KEYFRAME_MAX_GAP for gap in gaps)
    print(video_id, 'keyframes:', len(rows), '| frames đầu:', frame_ids[:10], '| gaps:', gaps[:10])
    print('Mapping đầu:', rows[:3])

In [ ]:
from IPython.display import display, Image
for video_id in videos:
    for row in builder.mapping_rows(video_id)[:3]:
        print(video_id, '| IMG', row['n'], '| FRAME', row['frame_idx'], '| TIME', row['pts_time'])
        display(Image(filename=str(builder.artifact_paths(video_id)['images'] / f"{row['n']:03d}.jpg"), width=360))

## 3. Sinh visual features bằng SigLIP

Dùng chính encoder/model revision và image preprocessing của `SiglipEncoder`, output float32 L2-normalized `(N,768)` vào `clip-features-32/<video_id>.npy`. Row thứ i khớp row thứ i của CSV, không dùng thứ tự filename kiểu lexicographic. Model chỉ load một lần cho stage này.

In [ ]:
builder.encode_visual_features(videos, batch_size=BATCH_SIZE)

## 4. Object detection trên CPU

Lưu `objects/<video_id>/001.json` đúng 5 field của corpus chính: detection_scores, detection_class_names, detection_class_entities, detection_boxes, detection_class_labels. Score/label/tọa độ là chuỗi; class_names là mã Open Images /m/...; box theo ymin,xmin,ymax,xmax normalized. Metadata/frame identity/checksum nằm riêng trong preparation/.

Detector CPU vẫn là SSDLite COCO; object_label_map.json chuyển các nhãn có ánh xạ chính xác sang mã/label Open Images chính thức. Nhãn không ánh xạ được bị bỏ qua. Không tạo prediction giả để đủ 100 detection. Chạy lại cell sẽ tự chuyển output cũ của pipeline sang schema này mà không cần chạy lại detector cho các ảnh đã có output.

In [ ]:
builder.generate_objects(videos, batch_size=BATCH_SIZE, threshold=0.2)

## 5. Caption + VQA bằng code hiện có

Gọi lại `generate_captions_with_mapping()` với BLIP caption-large và BLIP VQA-base pinned, `device='cpu'`, `num_workers=0`, batch 1. Output `caption_generator/<group>/<video_id>.json` gồm caption, vqa_answers, details, retrieval_text và identity/frame mapping.

Đây thường là bước chậm nhất trên CPU. Script lưu sau từng batch; nếu ngắt giữa chừng có thể chạy lại cell này để tiếp tục.

In [ ]:
builder.generate_captions(videos, batch_size=CAPTION_BATCH_SIZE)

## 6. Caption mapping, FAISS index và manifest

Gọi lại `build_caption_index()` với MiniLM pinned trên CPU. Sinh `caption_mapping/<group>/<video_id>.json`, `index/<group>/<video_id>.index` (384 chiều) và `index/<group>/<video_id>.meta.json` (SHA-256/model provenance). Chỉ xử lý các video đã chọn.

In [ ]:
builder.build_caption_indexes(videos, batch_size=8)

## 7. Validate toàn bộ artifact

Kiểm tra checksum, số row, frame_id/keyframe_n/image_path, object provenance, chiều vector và L2 normalization. Chỉ đưa dữ liệu vào corpus sau khi mọi video đều `OK`.

In [ ]:
import pandas as pd
from audit_artifact_formats import audit_video_formats
reports = [builder.validate_video(video_id) for video_id in videos]
display(pd.DataFrame(reports))
format_reports = [item for video_id in videos for item in audit_video_formats(OUTPUT_ROOT, video_id)]
display(pd.DataFrame(format_reports))
assert all(item['status'] == 'OK' for item in format_reports), format_reports
for video_id in videos:
    print(video_id)
    for kind, path in builder.artifact_paths(video_id).items():
        print(' ', kind, ':', path)

## 8. Đưa video mới vào hệ thống tìm kiếm

Sau smoke test, đổi `SMOKE_MODE=False` và chạy lại các cell để build **toàn bộ** video.

Nếu retrieval chạy trên máy khác: copy/merge `data/`, `mapping/`, `objects/`, `clip-features-32/`, `caption_generator/`, `caption_mapping/`, `index/` và `preparation/` sang đúng repo máy đó. File path được lưu tương đối nên chuyển máy được.

Tại máy đang chạy Milvus, chạy `python build_milvus.py rebuild` để corpus visual gồm cả video mới và cũ; nếu Milvus rỗng dùng `build`. Khởi động lại web app hoặc kernel retrieval để CaptionRetriever/BM25 nạp corpus mới.

Cell dưới chỉ chạy khi bạn bật `UPDATE_MILVUS=True`, Milvus đã chạy trên server này và môi trường có `pymilvus`. Việc tạo FAISS caption index ở bước 6 không cần Milvus.

In [ ]:
UPDATE_MILVUS = False
MILVUS_MODE = 'rebuild'  # 'build' nếu collection chưa tồn tại
if UPDATE_MILVUS:
    if SMOKE_MODE or OUTPUT_ROOT != PROJECT_DIR:
        raise RuntimeError('Không publish corpus từ smoke output.')
    import subprocess
    subprocess.run([sys.executable, str(PROJECT_DIR / 'build_milvus.py'), MILVUS_MODE], check=True)
else:
    print('Artifact đã sẵn sàng; cập nhật Milvus tại máy chạy retrieval sau full build.')